In [54]:
import numpy as np
import pandas as pd

In [55]:
df = pd.read_excel("C:/Users/vamsi/Downloads/Shop_trends_Raw_Dirty_Data.xlsx")
df.head()

,Customer_ID,Age,Gender,Location,Item Purchased,Category,Size,Color,Stock Level,Purchase Date,Quantity,Unit Price,Discount,Product Cost,Review Rating,Payment Method,Previous Purchases,Frequency of Purchases
0,1,55,Male,Kentucky,Blouse,Clothing,L,Gray,11,2025-01-03,4,53.0,0.25,33.95,3.1,Credit Card,14,Fortnightly
1,2,19,Male,Maine,Sweater,Clothing,L,Maroon,166,2025-02-26,5,64.0,0.25,28.68,3.1,Bank Transfer,2,Fortnightly
2,3,50,Male,Massachusetts,Jeans,Clothing,S,Maroon,205,2025-03-28,3,73.0,0.25,40.97,3.1,Cash,23,Weekly
3,4,21,Male,Rhode Island,Sandals,Footwear,M,Maroon,74,2025-04-02,5,90.0,0.05,41.18,3.5,PayPal,49,Weekly
4,5,45,Male,Oregon,Blouse,Clothing,M,Turquoise,92,2025-04-16,5,49.0,0.25,30.17,2.7,Cash,31,Annually


In [56]:
df.info() # Knowing abouy data
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 410 entries, 0 to 409
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Customer_ID             410 non-null    int64         
 1   Age                     410 non-null    int64         
 2   Gender                  410 non-null    object        
 3   Location                410 non-null    object        
 4   Item Purchased          410 non-null    object        
 5   Category                410 non-null    object        
 6   Size                    410 non-null    object        
 7   Color                   410 non-null    object        
 8   Stock Level             410 non-null    int64         
 9   Purchase Date           410 non-null    datetime64[ns]
 10  Quantity                410 non-null    int64         
 11  Unit Price              390 non-null    float64       
 12  Discount                410 non-null    float64   

,Customer_ID,Age,Stock Level,Purchase Date,Quantity,Unit Price,Discount,Product Cost,Review Rating,Previous Purchases
count,410.000000,410.000000,410.000000,410,410.000000,390.000000,410.000000,392.000000,410.000000,410.000000
mean,203.041463,44.421951,127.248780,2025-07-01 07:29:33.658536704,2.895122,59.633333,0.198293,32.288138,3.739512,26.248780
min,1.000000,18.000000,11.000000,2025-01-01 00:00:00,1.000000,20.000000,0.050000,8.380000,2.500000,1.000000
25%,102.250000,33.000000,65.250000,2025-04-04 06:00:00,2.000000,39.000000,0.200000,20.237500,3.100000,14.000000
50%,203.500000,45.000000,129.500000,2025-06-27 00:00:00,3.000000,59.500000,0.200000,31.370000,3.800000,28.000000
75%,303.750000,57.000000,187.000000,2025-09-27 00:00:00,4.000000,81.000000,0.250000,42.070000,4.300000,38.000000
max,402.000000,70.000000,249.000000,2025-12-28 00:00:00,5.000000,100.000000,0.250000,68.200000,5.000000,50.000000
std,116.477563,14.803522,69.241907,NaN,1.424970,23.647615,0.065483,14.081464,0.706657,14.321716


In [57]:
# Checking of Null Values
df.isnull()
df.isnull().sum() 

Customer_ID                0
Age                        0
Gender                     0
Location                   0
Item Purchased             0
Category                   0
Size                       0
Color                      0
Stock Level                0
Purchase Date              0
Quantity                   0
Unit Price                20
Discount                   0
Product Cost              18
Review Rating              0
Payment Method             0
Previous Purchases         0
Frequency of Purchases     0
dtype: int64

In [58]:
# Replacing Null values 
numeric_col = df.select_dtypes(include=["number"]).columns
for col in numeric_col:
    df[col] = df[col].fillna(df[col].mean().round(2))

In [59]:
#Checking & Removing of Duplicate values
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates()

Duplicates: 8


In [60]:
# Removing of Extra Space
categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()

In [61]:
# Fromating Date
df["Purchase Date"] = pd.to_datetime(df["Purchase Date"],format = "%d-%m-%Y")

In [62]:
df["Item Purchased"]= df["Item Purchased"].str.strip().str.title()

In [65]:
# Data Cleaning 
df["location"]= df["location"].str.strip().str.title()
df = df.rename(columns ={ "Item Purchased" : "product"}) # chaning Column Name

In [68]:
# Original Price for Every Product ( Quantity * Unit price) of each Product..
df["original_price"] = (df["quantity"] * df["unit_price"]).round(2)

In [69]:
# Finding the total Price for Each Product after Removing of discount value
Discount_Val = (
    df["original_price"] * df["discount"]
)

df["sale_price"] = (
    df["original_price"] - Discount_Val
).round(2)

In [70]:
# finding Profit Gain by Each Product 
df["profit"] = (
    df["sale_price"] - df["product_cost"]
).round(2)

In [79]:
df = df.rename(columns={ "frequency_of_purchases" : "Visit Frequency"}) #changing of Column name

In [80]:
# Providing Subscription for Customers Who Vist Weekly,Twice a week , Bi-weekly,Monthly 
df["Subscription"] = df["Visit Frequency"].replace(
    {
    "Weekly": "Silver Plan",
    "Fortnightly": "Gold Plan",
    "Bi-Weekly": "Basic Plan",
    "Monthly": "Starter Plan",
    "Every 3 Months":"No Subscription",
    "Quarterly":"No Subscription",
    "Annually":"No Subscription",
    
})

In [82]:
df["stock_remaining"] = ( 
    df["stock_level"] - df["quantity"]
)

In [83]:
# Contibution of Sale Among Male and Female
sales = (
    df.groupby("gender")["sale_price"].sum()
)
sales_contribution = (( sales / sales.sum()) * 100).round(2)
print(sales_contribution)

gender
Female    50.06
Male      49.94
Name: sale_price, dtype: float64


In [17]:
# Finding Category-Wise Sales
Category_sales = (
    df.groupby("Category")["Sale Price"]
    .sum().round(2)
)

print(Category_sales)

Category
Accessories    16175.31
Clothing       24758.40
Footwear        9082.13
Outerwear       6066.45
Name: Sale Price, dtype: float64


In [18]:
# Finding Most Common Visit Frequency by customers
cus_visit = (
    df.groupby("Visit Frequency")["Customer_ID"].count()
)
print( "Most of the Customers Visit For",cus_visit.idxmax())
print(" Visted Customers: ",cus_visit.max())

Most of the Customers Visit For Every 3 Months
 Visted Customers:  69


In [19]:
# Finding Most prefered Payment Method by Customers
payment_method = (
    df.groupby("Payment Method")["Customer_ID"].count()
)
print( "Most of the Customers Prefer to Pay by their",payment_method.idxmax())
print("prefer Customers: ",payment_method.max())

Most of the Customers Prefer to Pay by their Credit Card
prefer Customers:  73


In [20]:
# Finding Total Revenue 
revenue = df["Sale Price"].sum().round(2)
print("Total Revenue: ",revenue)

Total Revenue:  56082.28


In [21]:
# Finding Total Profit 
profit = df["Profit"].sum().round(2)
print("Total Profit: ",profit)

Total Profit:  43074.71


In [22]:
# Finding Total Quantity Sold
Quantity_sold = df["Quantity"].sum().round(2)
print("Total Quantity Sold:",Quantity_sold)

Total Quantity Sold: 1164


In [23]:
# Which Location Has Highest Sales 
Loc = (
    df.groupby("Location")["Sale Price"].sum().round(2)
)
print(Loc.idxmax(),"is the Highest Revenue Generated City") 
print("Generated Revenue:",Loc.max())

Mississippi is the Highest Revenue Generated City
Generated Revenue: 2232.5


In [24]:
# Which Location Has Recorded as  Highest profit Location
Loc = (
    df.groupby("Location")["Profit"].sum().round(2)
)
print(Loc.idxmax(),"is the Highest Profit Genetared City")
print("Generated Profit:",Loc.max())

Mississippi is the Highest Profit Genetared City
Generated Profit: 1759.43


In [25]:
# Monthly Sales
df["Month"] =(
    df["Purchase Date"].dt.month_name()
)
month_order = [
    "January", "February", "March",
    "April", "May", "June",
    "July", "August", "September",
    "October", "November", "December"
]
monthly_sales =(
    df.groupby("Month")["Sale Price"].sum().round(2)
)
monthly_sales = (
    monthly_sales.reindex(month_order)
)
print(monthly_sales)

Month
January      4255.60
February     3661.12
March        5024.35
April        5600.53
May          5885.61
June         4121.62
July         3469.10
August       6426.84
September    3989.68
October      4815.27
November     3462.00
December     5370.56
Name: Sale Price, dtype: float64


In [26]:
# Which Month Has Highest Sales
monthly_sales =(
    df.groupby("Month")["Sale Price"].sum().round(2)
)
print(monthly_sales.idxmax(),"is the Highest Revenue Generated Month")
print("Generated Revenue:",monthly_sales.max())

August is the Highest Revenue Generated Month
Generated Revenue: 6426.84


In [29]:
# which Category Generate High Revenue
category = (
    df.groupby("category")["sale_price"].sum()
)
print(category.idxmax(),"Generates the Highest Revenue")
print("Generated Revenue:",category.max())

Clothing Generates the Highest Revenue
Generated Revenue: 24758.4


In [30]:
# which Category Generate Highest Profit
category = (
    df.groupby("category")["profit"].sum().round(2)
)
print(category.idxmax(),"Generates the Highest profit")
print("Generated profit:",category.max())

Clothing Generates the Highest profit
Generated profit: 18668.15


In [31]:
# which Category Has High Rating
category = (
    df.groupby("category")["review_rating"].mean().round(2)
)
print(category.idxmax(),"has the High Rating Category")
print("Rating:",category.max())

Outerwear has the High Rating Category
Rating: 3.81


In [84]:
df["profit margin"] =(
    ((df["profit"] / df["sale_price"]) * 100).round(2)
)

In [64]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.lower()
)

In [34]:
df = df.drop("prouct", axis = 1)

In [85]:
df

,customer_id,age,gender,location,item_purchased,category,size,color,stock_level,purchase_date,...,payment_method,previous_purchases,Visit Frequency,Original Price,original_price,sale_price,profit,Subscription,stock_remaining,profit margin
0,1,55,Male,Kentucky,Blouse,Clothing,L,Gray,11,2025-01-03,...,Credit Card,14,Fortnightly,212.0,212.0,159.00,125.05,Gold Plan,7,78.65
1,2,19,Male,Maine,Sweater,Clothing,L,Maroon,166,2025-02-26,...,Bank Transfer,2,Fortnightly,320.0,320.0,240.00,211.32,Gold Plan,161,88.05
2,3,50,Male,Massachusetts,Jeans,Clothing,S,Maroon,205,2025-03-28,...,Cash,23,Weekly,219.0,219.0,164.25,123.28,Silver Plan,202,75.06
3,4,21,Male,Rhode Island,Sandals,Footwear,M,Maroon,74,2025-04-02,...,PayPal,49,Weekly,450.0,450.0,427.50,386.32,Silver Plan,69,90.37
4,5,45,Male,Oregon,Blouse,Clothing,M,Turquoise,92,2025-04-16,...,Cash,31,Annually,245.0,245.0,183.75,153.58,No Subscription,87,83.58
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
397,398,29,Male,Iowa,Blouse,Clothing,XL,Maroon,163,2025-08-04,...,Credit Card,3,Bi-Weekly,280.0,280.0,210.00,177.86,Basic Plan,158,84.70
398,399,63,Male,Louisiana,Shirt,Clothing,L,Black,108,2025-03-22,...,Venmo,35,Monthly,156.0,156.0,117.00,82.00,Starter Plan,106,70.09
399,400,60,Male,Alabama,Jewelry,Accessories,XL,Teal,107,2025-06-11,...,Bank Transfer,13,Bi-Weekly,25.0,25.0,20.00,4.94,Basic Plan,106,24.70
400,401,64,Male,Nevada,Backpack,Accessories,XL,Olive,136,2025-10-14,...,Credit Card,31,Annually,172.0,172.0,137.60,99.76,No Subscription,134,72.50


In [86]:
df.to_csv(
    "Shop_trends_Cleaned.csv",
    index=False
)